# ODE vs Discrete Forward Simulation Benchmarking
This notebook compares the mathematical accuracy and computational runtime of the High-Resolution ODE solver (`torchdiffeq.odeint`) against the highly-optimized native `ExperimentRunner`.

In [ ]:
import sys
import time
sys.path.append("..")

import torch
import matplotlib.pyplot as plt
from torchdiffeq import odeint
from src.config.schemas import ExperimentConfig, GameConfig, DynamicConfig, ExecutionConfig
from src.engine.runner import ExperimentRunner
from src.engine.statistics import load_experiment_stats
from src.dynamics.continuous import OMWUContinuous

plt.style.use('ggplot')

# Use CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)


In [ ]:
# ==========================================
# 1. Forward Discrete Simulation Benchmarking
# ==========================================
total_steps = 1000
eta = 0.05

config = ExperimentConfig(
    name="notebook_compare",
    game=GameConfig(generator="random", utility_range=(-1.0, 1.0), num_actions=[2, 2]),
    dynamic=DynamicConfig(
        algorithm="omwu",
        strict_theory_eta=False,
    ),
    execution=ExecutionConfig(total_steps=total_steps, device="auto",
        dtype="float32", # Can be float64 for higher precision steps_per_call=100, seed=42, compile=False)
)

runner = ExperimentRunner(config=config)

# Benchmark native runner
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0_discrete = time.time()

summary = runner.run(target_steps=total_steps)

torch.cuda.synchronize() if torch.cuda.is_available() else None
t1_discrete = time.time()
discrete_time = t1_discrete - t0_discrete

# Load exact regrets
stats_data = load_experiment_stats(output_dir=summary["output_dir"], session_id=summary["session_id"])
steps = stats_data["steps"].numpy()
discrete_regret_p1 = stats_data["cum_regrets"].numpy()[:, 0]  # Player 1
discrete_regret_p2 = stats_data["cum_regrets"].numpy()[:, 1]  # Player 2

# Extract the randomly generated game matrices
payoffs = runner.game.get_payoff_tensors()
U1 = payoffs[0].squeeze(0) if payoffs[0].dim() == 3 else payoffs[0]
U2 = payoffs[1].squeeze(0) if payoffs[1].dim() == 3 else payoffs[1]
A1, A2 = U1.shape


In [ ]:
# ==========================================
# 2. Forward Continuous ODE Benchmarking
# ==========================================
# We must use the exact empirical learning rate that the runner used
emp_eta = runner.dynamic.eta

if isinstance(emp_eta, torch.Tensor):
    emp_eta = emp_eta.item()

dyn = OMWUContinuous(U1, U2, emp_eta)
state0 = torch.zeros(2*A1 + 2*A2 + 2, device=device)
T = total_steps * emp_eta
t = torch.linspace(0, T, total_steps, device=device)

torch.cuda.synchronize() if torch.cuda.is_available() else None
t0_ode = time.time()

# Run ODE forward pass
states = odeint(dyn, state0, t, method='dopri5')

torch.cuda.synchronize() if torch.cuda.is_available() else None
t1_ode = time.time()
ode_time = t1_ode - t0_ode

# Extract continuous regret trajectory
idx = A1 + A2
Z1_traj = states[:, idx : idx+A1]; idx += A1
P1_traj = states[:, idx : idx+1]; idx += 1
Z2_traj = states[:, idx : idx+A2]; idx += A2
P2_traj = states[:, idx : idx+1]

continuous_regret_p1 = ((Z1_traj.max(dim=1).values - P1_traj.squeeze(1)) / emp_eta).cpu().numpy()
continuous_regret_p2 = ((Z2_traj.max(dim=1).values - P2_traj.squeeze(1)) / emp_eta).cpu().numpy()

print(f"Native Discrete Engine Time : {discrete_time:.4f} seconds")
print(f"Continuous ODE Forward Time : {ode_time:.4f} seconds")
print(f"ODE Overhead Ratio : {ode_time / discrete_time:.2f}x slower")


In [ ]:
# Plotting
import numpy as np
fig, axs = plt.subplots(1, 2, figsize=(14, 6))

# Plot Player 1
axs[0].plot(np.arange(1, total_steps+1), continuous_regret_p1, label="ODE Continuous Regret", linewidth=3, alpha=0.8)
axs[0].plot(steps, discrete_regret_p1, label="Discrete Native Regret", linestyle='--', linewidth=2, color='black')
axs[0].set_title("Continuous vs Discrete Regret (Player 1)")
axs[0].set_xlabel("Step k")
axs[0].set_ylabel("Cumulative Regret")
axs[0].legend()

# Plot Player 2
axs[1].plot(np.arange(1, total_steps+1), continuous_regret_p2, label="ODE Continuous Regret", linewidth=3, alpha=0.8, color='C1')
axs[1].plot(steps, discrete_regret_p2, label="Discrete Native Regret", linestyle='--', linewidth=2, color='black')
axs[1].set_title("Continuous vs Discrete Regret (Player 2)")
axs[1].set_xlabel("Step k")
axs[1].set_ylabel("Cumulative Regret")
axs[1].legend()

plt.tight_layout()
plt.show()
